## MAKE A BACKEND FOR MOVIE RECOMMENDATION ASSISTANT

In [1]:
#pip install -r ./requirements.txt

In [2]:
import pandas as pd

In [3]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

In [4]:
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore

In [5]:
df = pd.read_csv('./datas/cleaned_data.csv')
df.columns

Index(['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate', 'Genre',
       'IMDB_Rating', 'Overview', 'Director', 'Star1', 'Star2', 'Star3',
       'Star4', 'No_of_Votes', 'Runtime_min', 'age_group'],
      dtype='str')

### PUT THE DATAS IN VECTOR DB FOR SEMANTIC UNDERSTANDING

In [7]:
import hashlib

In [8]:
#Embed
documents = []
for _, row in df.iterrows():
    raw = f"{row['Series_Title']}_{row['Released_Year']}"
    movie_id = hashlib.md5(raw.encode()).hexdigest()
    content = f''' MovieName: {row['Series_Title']}
                    ReleasedYear: {row['Released_Year']}
                    IMDB_Rating: {row['IMDB_Rating']}
                    Synopsis: {row['Overview']}
                    Director: {row['Director']}
                    Actors: {row['Star1'], row['Star2'], row['Star3'], row['Star4']}
                    age_group: {row['age_group']}
'''
    doc = Document(page_content=content,
                   metadata={
                    'movie_id': movie_id,
                    'MovieName': row['Series_Title'],
                    'ReleasedYear': row['Released_Year'],
                    'Director': row['Director'],
                    'Actors': [row['Star1'], row['Star2'], row['Star3'], row['Star4']],
                    'IMDB_Rating': row['IMDB_Rating'],
                    'age_group': row['age_group']})
    documents.append(doc)

In [9]:
#Calculate cost
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    print(f'''Total Tokens: {total_tokens}
            Price(Rp.):{total_tokens/1000*0.00002*17200:.2f}''')
    
print_embedding_cost(documents)

Total Tokens: 102087
            Price(Rp.):35.12


In [10]:
os.environ["QDRANT_API_KEY"] = getpass.getpass("Enter your QDRANT API key: ")

In [11]:
os.environ["QDRANT_URL"] = getpass.getpass("Enter your QDRANT URL: ")

In [12]:
import os
import time
from uuid import uuid4
from qdrant_client import QdrantClient
from langchain_openai import OpenAIEmbeddings

client = QdrantClient(url=os.environ["QDRANT_URL"],api_key=os.environ.get("QDRANT_API_KEY"),
    timeout=120)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

collection_name = "movies"
batch_size = 50

# create collection once (if not exists)
client.recreate_collection( collection_name=collection_name,
    vectors_config={"size": 1536,"distance": "Cosine"})

for i in range(0, len(documents), batch_size):
    batch_docs = documents[i:i+batch_size]
    batch_texts = [doc.page_content for doc in batch_docs]
    batch_vectors = embeddings.embed_documents(batch_texts)
    points = []
    for j, doc in enumerate(batch_docs):
        points.append({
            "id": str(uuid4()),
            "vector": batch_vectors[j],
            "payload": {
                "page_content": doc.page_content,
                **doc.metadata}})

    client.upsert(collection_name=collection_name,points=points)

    print(f"Inserted {i} to {i + len(batch_docs)}")

    time.sleep(0.3)

C:\Users\MyBook Hype\AppData\Local\Temp\ipykernel_27424\908583236.py:16: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection( collection_name=collection_name,


Inserted 0 to 50
Inserted 50 to 100
Inserted 100 to 150
Inserted 150 to 200
Inserted 200 to 250
Inserted 250 to 300
Inserted 300 to 350
Inserted 350 to 400
Inserted 400 to 450
Inserted 450 to 500
Inserted 500 to 550
Inserted 550 to 600
Inserted 600 to 650
Inserted 650 to 700
Inserted 700 to 750
Inserted 750 to 800
Inserted 800 to 850
Inserted 850 to 900
Inserted 900 to 950
Inserted 950 to 1000


In [13]:
#Get all collection from Qdrant
colections_response = client.get_collections()

In [14]:
qdrant = QdrantVectorStore.from_existing_collection(embedding=embeddings, collection_name=collection_name,
                                                    url=os.environ["QDRANT_URL"],
                                                    api_key=os.environ['QDRANT_API_KEY'])

In [15]:
results = qdrant.similarity_search('most popular adult movies', k=5)
results

[Document(metadata={'_id': '7eb81207-5345-42e8-871c-e6bcb83b0cfe', '_collection_name': 'movies'}, page_content=" MovieName: Boogie Nights\n                    ReleasedYear: 1997\n                    IMDB_Rating: 7.9\n                    Synopsis: Back when sex was safe, pleasure was a business and business was booming, an idealistic porn producer aspires to elevate his craft to an art when he discovers a hot young talent.\n                    Director: Paul Thomas Anderson\n                    Actors: ('Mark Wahlberg', 'Julianne Moore', 'Burt Reynolds', 'Luis Guzmán')\n                    age_group: adults only\n"),
 Document(metadata={'_id': 'a48a2a28-e387-4154-b090-e9784f15e1bd', '_collection_name': 'movies'}, page_content=" MovieName: True Romance\n                    ReleasedYear: 1993\n                    IMDB_Rating: 7.9\n                    Synopsis: In Detroit, a lonely pop culture geek marries a call girl, steals cocaine from her pimp, and tries to sell it in Hollywood. Meanwh

In [16]:
##### FOR PARSING ######### and checking
for res in results:
    print(f'Result Page Content: {res.page_content}\n Result Metadata:[{res.metadata}]')
    print('────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────')

Result Page Content:  MovieName: Boogie Nights
                    ReleasedYear: 1997
                    IMDB_Rating: 7.9
                    Synopsis: Back when sex was safe, pleasure was a business and business was booming, an idealistic porn producer aspires to elevate his craft to an art when he discovers a hot young talent.
                    Director: Paul Thomas Anderson
                    Actors: ('Mark Wahlberg', 'Julianne Moore', 'Burt Reynolds', 'Luis Guzmán')
                    age_group: adults only

 Result Metadata:[{'_id': '7eb81207-5345-42e8-871c-e6bcb83b0cfe', '_collection_name': 'movies'}]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Result Page Content:  MovieName: True Romance
                    ReleasedYear: 1993
                    IMDB_Rating: 7.9
                    Synopsis: In Detroit, a lonely pop culture geek marries a call girl, steals cocaine from her pimp, and tries 